In [1]:
from pathlib import Path
import math
import warnings

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import seaborn as sns
from scipy import stats
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
import matplotlib.pyplot as plt
from sklearn.model_selection import (StratifiedKFold, cross_validate, train_test_split)
from sklearn.preprocessing import StandardScaler, OneHotEncoder, label_binarize
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, auc, classification_report, precision_score, recall_score,
                              f1_score, roc_auc_score, cohen_kappa_score, confusion_matrix, make_scorer,
                              ConfusionMatrixDisplay, roc_curve)
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
SEMILLA = 1669
OBJETIVO = "price"

In [2]:
# =============================================================================
# PARTE 1: CARGA, LIMPIEZA Y ANÁLISIS EXPLORATORIO
# =============================================================================

df = pd.read_csv(Path("Clean_Dataset.csv"), sep=",", decimal=".")

In [3]:
df

,Unnamed: 0,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
1,1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953
2,2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956
3,3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955
4,4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955
...,...,...,...,...,...,...,...,...,...,...,...,...
300148,300148,Vistara,UK-822,Chennai,Morning,one,Evening,Hyderabad,Business,10.08,49,69265
300149,300149,Vistara,UK-826,Chennai,Afternoon,one,Night,Hyderabad,Business,10.42,49,77105
300150,300150,Vistara,UK-832,Chennai,Early_Morning,one,Night,Hyderabad,Business,13.83,49,79099
300151,300151,Vistara,UK-828,Chennai,Early_Morning,one,Evening,Hyderabad,Business,10.00,49,81585


In [4]:
target_col = "class"
# Nota metodológica: se excluye 'price' de las variables predictoras porque
# el precio es casi determinístico respecto de la clase de cabina (las tarifas
# Business no se solapan con las de Economy), lo que generaría fuga de
# información (data leakage) y un problema trivial. Se conserva como variable
# de contexto para el análisis, pero no se usa como entrada del modelo.
feature_cols_num = ["duration", "days_left"]
feature_cols_cat = ["airline", "source_city", "departure_time", "stops",
                     "arrival_time", "destination_city"]

X = df[feature_cols_num + feature_cols_cat].copy()
y = (df[target_col] == "Business").astype(int)  # 1 = Business, 0 = Economy

class_counts = df[target_col].value_counts()
class_props = df[target_col].value_counts(normalize=True) * 100

In [5]:
# ---------------------------------------------------------------------------
# 2. Partición train / validation / test (60/20/20), estratificada
# ---------------------------------------------------------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=RANDOM_STATE, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp)

split_summary = pd.DataFrame({
    "Conjunto": ["Entrenamiento", "Validación", "Prueba"],
    "N° registros": [len(X_train), len(X_val), len(X_test)],
    "% del total": [len(X_train)/len(X)*100, len(X_val)/len(X)*100, len(X_test)/len(X)*100],
    "% Business": [y_train.mean()*100, y_val.mean()*100, y_test.mean()*100],
    "% Economy": [(1-y_train.mean())*100, (1-y_val.mean())*100, (1-y_test.mean())*100],
})

In [6]:
# ---------------------------------------------------------------------------
# 3. Preprocesamiento (codificación + escalado) ajustado SOLO con train
# ---------------------------------------------------------------------------
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), feature_cols_num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), feature_cols_cat),
])

X_train_enc = preprocessor.fit_transform(X_train)
X_val_enc = preprocessor.transform(X_val)
X_test_enc = preprocessor.transform(X_test)

n_features = X_train_enc.shape[1]

In [7]:
# ---------------------------------------------------------------------------
# 4. Técnicas de balanceo de clases (aplicadas SOLO sobre el set de entrenamiento)
# ---------------------------------------------------------------------------
balancing_sets = {}

# (a) Sin balanceo (referencia)
balancing_sets["Sin balanceo (baseline)"] = (X_train_enc, y_train)

# (b) Ponderación de clases (class_weight -> se aplica vía sample_weight)
balancing_sets["Ponderacion de clases (class_weight)"] = (X_train_enc, y_train)

# (c) Sobremuestreo aleatorio (RandomOverSampler)
ros = RandomOverSampler(random_state=RANDOM_STATE)
X_ros, y_ros = ros.fit_resample(X_train_enc, y_train)
balancing_sets["Sobremuestreo aleatorio (ROS)"] = (X_ros, y_ros)

# (d) Submuestreo aleatorio (RandomUnderSampler)
rus = RandomUnderSampler(random_state=RANDOM_STATE)
X_rus, y_rus = rus.fit_resample(X_train_enc, y_train)
balancing_sets["Submuestreo aleatorio (RUS)"] = (X_rus, y_rus)

balance_summary_rows = []
for name, (Xb, yb) in balancing_sets.items():
    n = len(yb)
    pct_bus = np.mean(yb) * 100
    balance_summary_rows.append([name, n, pct_bus, 100-pct_bus])
balance_summary = pd.DataFrame(balance_summary_rows,
    columns=["Tecnica", "N registros entrenamiento", "% Business", "% Economy"])

In [8]:
# ---------------------------------------------------------------------------
# 5. Arquitectura de la red neuronal (MLP) y entrenamiento con cada técnica
# ---------------------------------------------------------------------------
def build_model():
    return MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=60,
        early_stopping=True,
        n_iter_no_change=8,
        validation_fraction=0.1,
        random_state=RANDOM_STATE,
    )

results_rows = []
roc_data = {}
trained_models = {}

for name, (Xb, yb) in balancing_sets.items():
    model = build_model()

    if name.startswith("Ponderacion"):
        # MLPClassifier no soporta class_weight nativamente; se aproxima
        # duplicando el gradiente efectivo mediante sample_weight en un
        # wrapper manual no es directo en sklearn MLP, por lo que se
        # implementa balanceo por ponderación re-muestreando con pesos
        # (bootstrap ponderado) como aproximación práctica.
        weights = np.where(yb == 1, (yb == 0).sum() / len(yb), (yb == 1).sum() / len(yb))
        # bootstrap ponderado del tamaño original
        idx = np.random.choice(len(yb), size=len(yb), replace=True,
                                p=weights / weights.sum())
        Xb_use, yb_use = Xb[idx], yb.iloc[idx] if hasattr(yb, "iloc") else yb[idx]
    else:
        Xb_use, yb_use = Xb, yb

    model.fit(Xb_use, yb_use)
    trained_models[name] = model

    val_pred = model.predict(X_val_enc)
    val_proba = model.predict_proba(X_val_enc)[:, 1]

    acc = accuracy_score(y_val, val_pred)
    prec = precision_score(y_val, val_pred)
    rec = recall_score(y_val, val_pred)
    f1 = f1_score(y_val, val_pred)
    auc = roc_auc_score(y_val, val_proba)
    n_iter = model.n_iter_
    final_loss = model.loss_

    results_rows.append([name, acc, prec, rec, f1, auc, n_iter, final_loss])

    fpr, tpr, _ = roc_curve(y_val, val_proba)
    roc_data[name] = (fpr, tpr, auc)

results_df = pd.DataFrame(results_rows, columns=[
    "Tecnica", "Exactitud", "Precision", "Recall", "F1-score", "AUC-ROC",
    "N iteraciones", "Perdida final"])

best_name = results_df.sort_values("F1-score", ascending=False).iloc[0]["Tecnica"]
best_model = trained_models[best_name]

# Evaluación final sobre el conjunto de prueba con el mejor modelo
test_pred = best_model.predict(X_test_enc)
test_proba = best_model.predict_proba(X_test_enc)[:, 1]
test_metrics = {
    "Exactitud": accuracy_score(y_test, test_pred),
    "Precision": precision_score(y_test, test_pred),
    "Recall": recall_score(y_test, test_pred),
    "F1-score": f1_score(y_test, test_pred),
    "AUC-ROC": roc_auc_score(y_test, test_proba),
}

In [13]:
# ---------------------------------------------------------------------------
# 6. Gráficos
# ---------------------------------------------------------------------------
plt.rcParams.update({"font.size": 10})

# 6.1 Distribución original de clases
fig, ax = plt.subplots(figsize=(5, 4))
class_counts.plot(kind="bar", color=["#8b1e2f", "#c98a94"], ax=ax)
ax.set_title("Distribución original de la variable 'class'")
ax.set_xlabel("Clase")
ax.set_ylabel("N° de registros")
for i, v in enumerate(class_counts.values):
    ax.text(i, v + 2000, f"{v:,}", ha="center")
plt.tight_layout()
plt.savefig(f"./fig_distribucion_original.png", dpi=150)
plt.close()

# 6.2 Efecto de las técnicas de balanceo (barras comparativas)
fig, ax = plt.subplots(figsize=(6.5, 4))
x = np.arange(len(balance_summary))
ax.bar(x, balance_summary["% Economy"], label="Economy", color="#c98a94")
ax.bar(x, balance_summary["% Business"], bottom=balance_summary["% Economy"],
       label="Business", color="#8b1e2f")
ax.set_xticks(x)
ax.set_xticklabels(balance_summary["Tecnica"], rotation=20, ha="right", fontsize=8)
ax.set_ylabel("% de registros en entrenamiento")
ax.set_title("Efecto de las técnicas de balanceo sobre el set de entrenamiento")
ax.legend()
plt.tight_layout()
plt.savefig(f"./fig_balanceo_tecnicas.png", dpi=150)
plt.close()

# 6.3 Curvas de pérdida (loss curve) de cada modelo
fig, ax = plt.subplots(figsize=(6.5, 4))
for name, model in trained_models.items():
    ax.plot(model.loss_curve_, label=name, linewidth=1.5)
ax.set_xlabel("Épocas")
ax.set_ylabel("Pérdida (log-loss)")
ax.set_title("Curvas de convergencia del entrenamiento por técnica")
ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(f"./fig_curvas_perdida.png", dpi=150)
plt.close()

# 6.4 Comparación de métricas por técnica (barras agrupadas)
fig, ax = plt.subplots(figsize=(7, 4.2))
metrics_to_plot = ["Exactitud", "Precision", "Recall", "F1-score", "AUC-ROC"]
x = np.arange(len(results_df))
width = 0.15
colors = ["#8b1e2f", "#c98a94", "#4a4a4a", "#d4af37", "#5b7f95"]
for i, m in enumerate(metrics_to_plot):
    ax.bar(x + i*width, results_df[m], width, label=m, color=colors[i])
ax.set_xticks(x + width*2)
ax.set_xticklabels(results_df["Tecnica"], rotation=20, ha="right", fontsize=7.5)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Valor de la métrica")
ax.set_title("Comparación de métricas en el conjunto de validación")
ax.legend(fontsize=7, ncol=3)
plt.tight_layout()
plt.savefig(f"./fig_comparacion_metricas.png", dpi=150)
plt.close()

# 6.5 Curvas ROC
fig, ax = plt.subplots(figsize=(5.5, 4.5))
for name, (fpr, tpr, auc) in roc_data.items():
    ax.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})", linewidth=1.5)
ax.plot([0, 1], [0, 1], "k--", linewidth=1)
ax.set_xlabel("Tasa de falsos positivos")
ax.set_ylabel("Tasa de verdaderos positivos")
ax.set_title("Curvas ROC por técnica de balanceo (validación)")
ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(f"./fig_roc_curves.png", dpi=150)
plt.close()

# 6.6 Matriz de confusión del mejor modelo sobre test
fig, ax = plt.subplots(figsize=(4.5, 4.5))
cm = confusion_matrix(y_test, test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Economy", "Business"])
disp.plot(ax=ax, cmap="Reds", colorbar=False)
ax.set_title(f"Matriz de confusión - conjunto de prueba\n({best_name})")
plt.tight_layout()
plt.savefig(f"./fig_matriz_confusion.png", dpi=150)
plt.close()

In [14]:
# ---------------------------------------------------------------------------
# 7. Exportar tablas a CSV para insertarlas en el informe
# ---------------------------------------------------------------------------
split_summary.round(2).to_csv(f"./tabla_split.csv", index=False)
balance_summary.round(2).to_csv(f"./tabla_balanceo.csv", index=False)
results_df.round(4).to_csv(f"./tabla_resultados.csv", index=False)
pd.DataFrame([test_metrics]).round(4).to_csv(f"./tabla_test_final.csv", index=False)

print("SPLIT SUMMARY")
print(split_summary)
print("\nBALANCE SUMMARY")
print(balance_summary)
print("\nRESULTS (validation)")
print(results_df)
print("\nBEST MODEL:", best_name)
print("\nTEST METRICS (best model):")
print(test_metrics)
print("\nN features after encoding:", n_features)

SPLIT SUMMARY
        Conjunto  N° registros  % del total  % Business  % Economy
0  Entrenamiento        180091    59.999733   31.146476  68.853524
1     Validación         60031    20.000133   31.147241  68.852759
2         Prueba         60031    20.000133   31.145575  68.854425

BALANCE SUMMARY
                                Tecnica  N registros entrenamiento  \
0               Sin balanceo (baseline)                     180091   
1  Ponderacion de clases (class_weight)                     180091   
2         Sobremuestreo aleatorio (ROS)                     247998   
3           Submuestreo aleatorio (RUS)                     112184   

   % Business  % Economy  
0   31.146476  68.853524  
1   31.146476  68.853524  
2   50.000000  50.000000  
3   50.000000  50.000000  

RESULTS (validation)
                                Tecnica  Exactitud  Precision    Recall  \
0               Sin balanceo (baseline)   0.704136   0.535228  0.380682   
1  Ponderacion de clases (class_weight)   0